In [ ]:
from loguru import logger
import sys
import os
import re
from pymongo import MongoClient
path = r"C:\Users\Admin\Documents\V03-120126"
sys.path.append(path)
from constants import MongoDBConfig, MinioConfig, MigrateConfig, MongoDBCollectionConfig

CREATE_BY = ["SYSTEM", "V03"]

# ================== CLIENT INIT ==================
mongo_client = MongoClient(
    host=MongoDBConfig.HOST,
    port=MongoDBConfig.PORT,
    username=MongoDBConfig.USERNAME,
    password=MongoDBConfig.PASSWORD
)

source_db = mongo_client['v03_core_301225_v0']
clone_db = mongo_client['v03_core_301225_v2']

In [ ]:

def clone_database():
    # Get all collection names
    collection_names = source_db.list_collection_names()
    
    logger.info(f"Found {len(collection_names)} collections to clone: {collection_names}")
    
    for collection_name in collection_names:
        # Skip system collections if any
        if collection_name.startswith("system."):
            continue

        logger.info(f"Starting clone for collection: {collection_name}")
        
        source_col = source_db[collection_name]
        target_col = clone_db[collection_name]
        
        # Optional: Clear target collection before cloning
        # target_col.drop()
        # logger.info(f"Dropped existing collection: {collection_name} in target DB")

        doc_count = source_col.count_documents({})
        if doc_count == 0:
            logger.warning(f"Collection {collection_name} is empty. Skipping.")
            continue

        batch_size = 1000
        batch = []
        count = 0
        
        # Cursor to iterate over documents
        # If you need to filter by CREATE_BY, use: {"created_by": {"$in": CREATE_BY}}
        cursor = source_col.find({})
        
        for doc in cursor:
            batch.append(doc)
            if len(batch) >= batch_size:
                try:
                    target_col.insert_many(batch)
                    count += len(batch)
                    logger.info(f"Cloned {count}/{doc_count} documents for {collection_name}")
                    batch = []
                except Exception as e:
                    logger.error(f"Error inserting batch into {collection_name}: {e}")

        # Insert remaining documents
        if batch:
            try:
                target_col.insert_many(batch)
                count += len(batch)
                logger.info(f"Cloned {count}/{doc_count} documents for {collection_name}")
            except Exception as e:
                logger.error(f"Error inserting remaining batch into {collection_name}: {e}")
                
        logger.success(f"Finished cloning collection: {collection_name}")

    logger.success("Database clone process completed.")

if __name__ == "__main__":
    clone_database()
